# 02 · Rate of clinical progression by baseline SAA status

Linear mixed models of UPDRS III, Hoehn and Yahr and MoCA on time × baseline SAA status (SAA-negative = reference), in all prodromal participants.

Phenoconversion during follow-up (PRIMDIAG = 1 or 5 at any visit) is handled two ways:

1. **adjust** – converters kept; `conv` and `time:conv` entered as covariates → `analysis/lmm_saa_status_conv_adj_bladj/`
2. **exclude** – converters removed entirely (sensitivity analysis) → `analysis/lmm_saa_status_excl_conv_bladj/`

Requires `results_baseline/prodromal_saa_longitudinal.csv` from notebook 01.

In [ ]:
import os
BASE_DIR = os.environ.get('PPMI_SAA_BASE',
                          '/PPMI_Projects/prodromal_SAA_groups')

## Configuration

In [ ]:
# ------------------------------- CONFIG -------------------------------------
# Output folder for each way of handling phenoconversion (kept separate so the
# two analyses never overwrite each other).
LMM_OUT_NAMES = {
    'adjust':  'lmm_saa_status_conv_adj_bladj',    # converters kept, `conv` is a covariate
    'exclude': 'lmm_saa_status_excl_conv_bladj',   # converters removed (sensitivity analysis)
}

# Grouping variable: baseline SAA status. Reference = Negative.
LMM_POS_LABEL = 'Positive'
LMM_NEG_LABEL = 'Negative'          # <-- REFERENCE

# Phenoconversion during follow-up, defined from PRIMDIAG
LMM_CONV_COL   = 'PRIMDIAG'
LMM_CONV_CODES = [1, 5]             # 1 = idiopathic PD, 5 = DLB
LMM_CONV_TIME_INTERACTION = True    # 'adjust' only: also add time:conv (adjusts the slope)

# Longitudinal outcomes: (column, display name, higher score means)
LMM_OUTCOMES = [
    ('updrs3_score', 'UPDRS III motor score', 'worse'),
    ('hy',           'Hoehn and Yahr stage',  'worse'),
    ('moca',         'MoCA score',            'better'),
    # ('pigd',       'PIGD score',            'worse'),
]

LMM_TIME_COL      = 'years_from_BL_exact'
LMM_TIME_MAX      = 12       # fit and plot only the first N years
LMM_MIN_VISITS    = 2        # a participant needs this many outcome values
LMM_MIN_PER_GROUP = 5        # minimum participants per SAA group for a model to be fitted

# Fixed effects besides time and group. Any outcome NOT listed in
# LMM_OUTCOME_COVARIATES uses LMM_COVARIATES. A term can be a column name
# ('EDUCYRS') or a categorical term ('C(race)').
# In the 'adjust' analysis `conv` is inserted automatically (just before the
# outcome's own baseline term, bl_<outcome>), plus time:conv if requested above.
LMM_COVARIATES = ['age_bl', 'sex']
LMM_OUTCOME_COVARIATES = {
    'moca':         ['age_bl', 'sex', 'EDUCYRS', 'C(fampd_bin)', 'APOE_e4', 'bl_moca'],
    'hy':           ['age_bl', 'sex', 'C(fampd_bin)', 'bl_hy'],
    # 'pigd':       ['age_bl', 'sex', 'C(fampd_bin)', 'bl_pigd'],
    'updrs3_score': ['age_bl', 'sex', 'C(fampd_bin)', 'bl_updrs3_score'],
}
LMM_RANDOM_SLOPE = True      # random intercept + slope; False = intercept only

LMM_MT    = 'holm'           # correction across the outcomes
LMM_ALPHA = 0.05
LMM_CPOS, LMM_CNEG = '#E58A3A', '#1D9E75'     # SAA-positive / SAA-negative (ref)
LMM_DPI = 600

# Baseline value of every outcome is built as column bl_<outcome> (bl_moca, bl_hy,
# bl_pigd, bl_updrs3_score). To adjust for it, add it to that outcome's list above
# ('time:bl_hy' would also adjust the slope). bl_* covariates are mean-centred inside
# each model. When an outcome's OWN baseline is in its list, the visit that supplied
# that value is removed from the response (ANCOVA-type LMM); otherwise that row would
# fit its own covariate exactly and the fit fails (LinAlgError).
LMM_BL_WINDOW = 0.5          # years: baseline = non-missing value closest to time 0 within this
LMM_BL_MIN_FOLLOWUP = 1      # post-baseline values needed when an outcome is baseline-adjusted

LMM_YEAR_BIN = 'round'       # n-per-year tables: 'round' = nearest year, 'floor' = completed years
# ---------------------------- END OF CONFIG ---------------------------------

## Functions

In [ ]:
from pathlib import Path
import warnings
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings('ignore')

LMM_BASE = Path(BASE_DIR)
LMM_SRC  = LMM_BASE / 'results_baseline' / 'prodromal_saa_longitudinal.csv'
if not LMM_SRC.exists():
    raise FileNotFoundError(f'Missing input: {LMM_SRC}  (run 01_build_dataset first)')

plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': LMM_DPI, 'savefig.bbox': 'tight',
                     'font.size': 12, 'axes.titlesize': 12, 'axes.labelsize': 12,
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.alpha': 0.22, 'grid.linewidth': 0.5,
                     'pdf.fonttype': 42, 'ps.fonttype': 42})

LMM_MODES = ('adjust', 'exclude')


# ---------------------------------------------------------------------------
# Small helpers
# ---------------------------------------------------------------------------
def lmm_p(p):
    if p is None or not np.isfinite(p):
        return ''
    return '<0.001' if p < 0.001 else f'{p:.3f}'


def lmm_ptxt(p):
    s = lmm_p(p)
    if s == '':
        return 'P = NA'
    return 'P' + s if s.startswith('<') else 'P = ' + s


def lmm_stars(p):
    if p is None or not np.isfinite(p):
        return ''
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''


def lmm_fmt(est, lo, hi, dec=3):
    if est is None or not np.isfinite(est):
        return 'not estimable'
    return f'{est:.{dec}f} ({lo:.{dec}f} to {hi:.{dec}f})'


def lmm_covs_for(outcome, mode):
    """Covariate terms for one outcome. In 'adjust' mode `conv` is inserted just
    before the outcome's own baseline term (or appended if there is none)."""
    covs = list(LMM_OUTCOME_COVARIATES.get(outcome, LMM_COVARIATES))
    if mode == 'adjust' and 'conv' not in covs:
        bl = f'bl_{outcome}'
        if bl in covs:
            covs.insert(covs.index(bl), 'conv')
        else:
            covs.append('conv')
    return covs


def lmm_term_col(term):
    """Data column behind a covariate term: 'C(race)' -> 'race', 'EDUCYRS' -> 'EDUCYRS'."""
    term = term.strip()
    if term.startswith('time:'):                 # 'time:bl_hy' -> 'bl_hy'
        term = term[5:].strip()
    mt = re.fullmatch(r'\s*C\(\s*([A-Za-z_]\w*)\s*(,.*)?\)\s*', term)
    return mt.group(1) if mt else term.strip()


def lmm_terms_txt(outcome, mode):
    """Covariate part of the formula for one outcome, as printed."""
    covs = lmm_covs_for(outcome, mode)
    if mode == 'adjust' and LMM_CONV_TIME_INTERACTION and 'conv' in covs:
        covs = covs + ['time:conv']
    return ' + '.join(covs)


def lmm_bl_adj(outcome):
    """True if this outcome's own baseline (bl_<outcome>) is in its covariate list."""
    return f'bl_{outcome}' in lmm_covs_for(outcome, 'exclude')


# every covariate key must be an outcome -- stop here instead of silently ignoring it
lmm_bad_keys = [k for k in LMM_OUTCOME_COVARIATES if k not in [oc for oc, _n, _d in LMM_OUTCOMES]]
if lmm_bad_keys:
    raise KeyError(f'LMM_OUTCOME_COVARIATES has keys that are not outcomes: {lmm_bad_keys}')


# ---------------------------------------------------------------------------
# Number of participants at each follow-up year (from the data in each model)
# ---------------------------------------------------------------------------
def lmm_year_table(results, group_col, group_labels, bin_mode=LMM_YEAR_BIN):
    """results: list of (outcome display name, fitted result dict)."""
    blocks = []
    for oname, r in results:
        if '_data' not in r:
            continue
        d = r['_data'].copy()
        yr = np.round(d['time']) if bin_mode == 'round' else np.floor(d['time'])
        d['Year'] = yr.astype(int)
        t = (d.groupby(['Year', group_col]).PATNO.nunique()
             .unstack(fill_value=0).reindex(columns=[0.0, 1.0], fill_value=0))
        # last row = unique participants in the model (NOT the column sum)
        t.loc['Total (model n)'] = [d.loc[d[group_col] == 0, 'PATNO'].nunique(),
                                    d.loc[d[group_col] == 1, 'PATNO'].nunique()]
        t.columns = pd.MultiIndex.from_product([[oname], list(group_labels)])
        blocks.append(t)
    tab = pd.concat(blocks, axis=1).fillna(0).astype(int)
    tab.index.name = 'Year'
    return tab


def lmm_year_md(tab):
    cols = [f'{a}, {b}' for a, b in tab.columns]
    out = ['| Year | ' + ' | '.join(cols) + ' |', '|---' * (len(cols) + 1) + '|']
    for idx, row in tab.iterrows():
        out.append(f'| {idx} | ' + ' | '.join(str(v) for v in row.values) + ' |')
    return '\n'.join(out)


def lmm_n_per_year(run):
    """Table of participants per follow-up year, by SAA status and outcome."""
    tab = lmm_year_table([(on, run['res'][oc]) for oc, on, _ in LMM_OUTCOMES],
                         'saa', ('SAA-neg', 'SAA-pos'))
    print('\n=== All prodromal participants ===')
    print(tab.to_string())
    tab.to_csv(run['out'] / 'lmm_saa_n_per_year_Overall.csv')
    (run['out'] / 'lmm_saa_n_per_year_Overall.md').write_text(lmm_year_md(tab), encoding='utf-8')
    print('\nwritten to', run['out'])


def lmm_n_flow(run):
    """Participant flow from the longitudinal file to the models."""
    used = set().union(*[set(run['res'][oc]['_data'].PATNO)
                         for oc, _, _ in LMM_OUTCOMES if '_data' in run['res'][oc]])
    n_all = run['n_all']           # longitudinal dataset (<= LMM_TIME_MAX years)
    n_saa = run['n_saa']           # with a baseline SAA status
    n_cov = run['n_analysed']      # after converter exclusion (= n_saa when adjusting)
    print('Longitudinal dataset           :', n_all)
    print('Excluded, no baseline SAA      :', n_all - n_saa)
    if run['mode'] == 'exclude':
        print('Excluded, phenoconverters      :', n_saa - n_cov)
    print('Excluded, insufficient outcome :', n_cov - len(used))
    print('LMM modeling (any outcome)     :', len(used))


# ---------------------------------------------------------------------------
# Main analysis
#   mode = 'adjust'  : converters kept, conv (+ time:conv) entered as covariates
#   mode = 'exclude' : converters removed entirely before any model is fitted
# ---------------------------------------------------------------------------
def run_lmm_saa(mode):
    if mode not in LMM_MODES:
        raise ValueError(f'mode must be one of {LMM_MODES}')
    adjust = mode == 'adjust'
    LMM_OUT = LMM_BASE / 'analysis' / LMM_OUT_NAMES[mode]
    LMM_OUT.mkdir(parents=True, exist_ok=True)

    # ---------------------------- data ----------------------------
    lmm_d = pd.read_csv(LMM_SRC, na_values=[''], low_memory=False).sort_values(['PATNO', 'visit_n'])
    lmm_d['time'] = pd.to_numeric(lmm_d[LMM_TIME_COL], errors='coerce')
    if LMM_TIME_MAX is not None:
        lmm_d = lmm_d[lmm_d.time <= LMM_TIME_MAX]

    lmm_bl = lmm_d[lmm_d.is_baseline == 1].set_index('PATNO')

    # SAA status -> 1 / 0 ; anything else is dropped
    lmm_stat = lmm_bl['SAA_Status_BL'].astype(str).str.strip().str.capitalize()
    lmm_saa = pd.Series(np.nan, index=lmm_bl.index, dtype=float)
    lmm_saa[lmm_stat == LMM_POS_LABEL.capitalize()] = 1.0
    lmm_saa[lmm_stat == LMM_NEG_LABEL.capitalize()] = 0.0

    # phenoconversion: PRIMDIAG in LMM_CONV_CODES at ANY visit within the
    # (already time-capped) follow-up window -> 1, otherwise 0
    if LMM_CONV_COL not in lmm_d.columns:
        raise KeyError(f'Column {LMM_CONV_COL} not found in {LMM_SRC.name}')
    lmm_pdx = pd.to_numeric(lmm_d[LMM_CONV_COL], errors='coerce')
    lmm_conv_ids = set(lmm_d.loc[lmm_pdx.isin(LMM_CONV_CODES), 'PATNO'])
    lmm_conv = pd.Series(lmm_bl.index.isin(lmm_conv_ids).astype(float), index=lmm_bl.index)

    # baseline covariates, carried onto every visit row
    lmm_cov = pd.DataFrame({
        'age_bl': pd.to_numeric(lmm_bl.get('age_at_visit'), errors='coerce'),
        'sex': pd.to_numeric(lmm_bl.get('SEX'), errors='coerce'),
        'conv': lmm_conv,
        'saa': lmm_saa,
    })
    lmm_cov = lmm_cov[lmm_cov.saa.notna()]
    lmm_cov = lmm_cov[~lmm_cov.index.duplicated(keep='first')]
    n_saa = len(lmm_cov)

    # converters by SAA status (counted before any exclusion)
    conv_word = 'converters' if adjust else 'excluded'
    lmm_conv_tab = pd.DataFrame(
        {f'{conv_word}, SAA-negative': [int(lmm_cov.loc[lmm_cov.saa == 0, 'conv'].sum())],
         f'{conv_word}, SAA-positive': [int(lmm_cov.loc[lmm_cov.saa == 1, 'conv'].sum())]},
        index=pd.Index(['Overall'], name='group'))
    lmm_excl_ids = []
    if adjust:
        lmm_conv_tab.to_csv(LMM_OUT / 'lmm_saa_conversion_counts.csv')
    else:
        lmm_conv_tab.to_csv(LMM_OUT / 'lmm_saa_excluded_converters.csv')
        lmm_excl_ids = sorted(lmm_cov.index[lmm_cov.conv == 1])
        pd.Series(lmm_excl_ids, name='PATNO').to_csv(
            LMM_OUT / 'lmm_saa_excluded_converters_ids.csv', index=False)
        # EXCLUDE converters entirely (all visits, both SAA groups)
        lmm_cov = lmm_cov[lmm_cov.conv == 0].drop(columns='conv')

    lmm_long = lmm_d[lmm_d.PATNO.isin(lmm_cov.index)].copy()
    # the longitudinal file may already carry columns with these names (e.g. sex);
    # drop them so the baseline-derived versions are the ones used
    lmm_long = lmm_long.drop(columns=[c for c in lmm_cov.columns if c in lmm_long.columns])
    lmm_long = lmm_long.join(lmm_cov, on='PATNO')

    # baseline value of each outcome (bl_<outcome>): non-missing value closest to time 0
    LMM_BL_ROWS = {}
    for _oc, _on, _dr in LMM_OUTCOMES:
        if _oc not in lmm_long.columns:
            continue
        _v = lmm_long[['PATNO', 'time', _oc]].copy()
        _v[_oc] = pd.to_numeric(_v[_oc], errors='coerce')
        _v['_at'] = _v.time.abs()
        _v = _v[(_v._at <= LMM_BL_WINDOW) & _v[_oc].notna()].sort_values(['PATNO', '_at'])
        _first = _v.groupby('PATNO', sort=False).head(1)
        LMM_BL_ROWS[_oc] = set(_first.index)      # used only if the outcome is baseline-adjusted
        _blv = _first.set_index('PATNO')[_oc].rename(f'bl_{_oc}')
        lmm_long = lmm_long.drop(columns=[f'bl_{_oc}'], errors='ignore').join(_blv, on='PATNO')

    # every covariate column must exist -- stop here instead of silently dropping it
    lmm_missing = sorted({lmm_term_col(t) for oc, _n, _d in LMM_OUTCOMES
                          for t in lmm_covs_for(oc, mode)} - set(lmm_long.columns))
    if lmm_missing:
        raise KeyError(f'Covariate column(s) not found in the data: {lmm_missing}')

    # group sizes
    lmm_sizes = pd.DataFrame({'SAA-negative': [int((lmm_cov.saa == 0).sum())],
                              'SAA-positive': [int((lmm_cov.saa == 1).sum())]},
                             index=pd.Index(['Overall'], name='group'))
    lmm_sizes.to_csv(LMM_OUT / 'lmm_saa_group_sizes.csv')

    if adjust:
        print(f'Participants with a baseline SAA status: {len(lmm_cov)}   visit rows: {len(lmm_long)}')
    else:
        print(f'Converters excluded ({LMM_CONV_COL} in {LMM_CONV_CODES} at any visit, '
              f'<= {LMM_TIME_MAX} y): {len(lmm_excl_ids)} of {n_saa} participants')
        print(f'Participants analysed (non-converters): {len(lmm_cov)}   visit rows: {len(lmm_long)}')
    print('Model: outcome ~ time * SAA + <covariates> + '
          + ('(1 + time | PATNO)' if LMM_RANDOM_SLOPE else '(1 | PATNO)')
          + '   [reference group = SAA-negative]')
    for oc, oname, _d in LMM_OUTCOMES:
        print(f'   covariates, {oname:<22}: {lmm_terms_txt(oc, mode)}')
    if adjust:
        print(f'conv = 1 if {LMM_CONV_COL} in {LMM_CONV_CODES} at any visit '
              f'(<= {LMM_TIME_MAX} y): {int(lmm_cov.conv.sum())} of {len(lmm_cov)} participants')
    if adjust:
        print('\nBaseline group sizes')
        print(lmm_sizes.to_string(), '\n')
        print('Phenoconverters during follow-up')
        print(lmm_conv_tab.to_string(), '\n')
    else:
        print('\nConverters excluded')
        print(lmm_conv_tab.to_string(), '\n')
        print('Baseline group sizes after exclusion')
        print(lmm_sizes.to_string(), '\n')

    # ---------------------------- fit ----------------------------
    def lmm_blank(outcome, note, n_neg=0, n_pos=0, n_obs=0):
        return {'outcome': outcome, 'note': note,
                'n_participants': int(n_neg + n_pos), 'n_observations': int(n_obs),
                'n_neg': int(n_neg), 'n_pos': int(n_pos),
                'slope_neg': np.nan, 'slope_neg_lo': np.nan, 'slope_neg_hi': np.nan,
                'slope_pos': np.nan, 'slope_pos_lo': np.nan, 'slope_pos_hi': np.nan,
                'diff': np.nan, 'diff_lo': np.nan, 'diff_hi': np.nan, 'p': np.nan,
                'p_adj': np.nan}

    def lmm_fit_one(dat, outcome):
        """Fit one LMM of `outcome` on time x SAA status. Reference = SAA-negative."""
        covs_all = lmm_covs_for(outcome, mode)     # covariates for THIS outcome
        cols = ['PATNO', 'time', outcome, 'saa'] + [lmm_term_col(t) for t in covs_all]
        d = dat[[c for c in dict.fromkeys(cols) if c in dat.columns]].copy()
        if outcome not in d.columns:
            return lmm_blank(outcome, 'column not in the dataset')
        d[outcome] = pd.to_numeric(d[outcome], errors='coerce')
        if lmm_bl_adj(outcome):                     # own baseline -> covariate, not response
            d = d[~d.index.isin(LMM_BL_ROWS.get(outcome, ()))]
        d = d.dropna()
        if d.empty:
            return lmm_blank(outcome, 'no complete observations')
        cnt = d.groupby('PATNO')[outcome].size()
        d = d[d.PATNO.isin(cnt[cnt >= (LMM_BL_MIN_FOLLOWUP if lmm_bl_adj(outcome)
                                       else LMM_MIN_VISITS)].index)]

        n_neg = int(d.loc[d.saa == 0, 'PATNO'].nunique())
        n_pos = int(d.loc[d.saa == 1, 'PATNO'].nunique())
        if n_neg < LMM_MIN_PER_GROUP or n_pos < LMM_MIN_PER_GROUP:
            return lmm_blank(outcome,
                             f'too few participants (neg = {n_neg}, pos = {n_pos})',
                             n_neg, n_pos, len(d))
        d = d.rename(columns={outcome: 'y'})

        if adjust:
            # covariates with no variation (e.g. no converters) are dropped
            covs = [t for t in covs_all if d[lmm_term_col(t)].nunique() > 1]
            dropped = [t for t in covs_all if t not in covs]
            terms = covs + (['time:conv'] if LMM_CONV_TIME_INTERACTION and 'conv' in covs else [])
            drop_note = f'dropped (no variation): {", ".join(dropped)}' if dropped else ''
        else:
            terms, drop_note = covs_all, ''

        for _c in [c for c in d.columns if c.startswith('bl_')]:   # centre baseline covariates
            d[_c] = d[_c] - d[_c].mean()
        formula = 'y ~ time * saa' + (' + ' + ' + '.join(terms) if terms else '')
        note, res = '', None
        for re_formula in (['~time', None] if LMM_RANDOM_SLOPE else [None]):
            try:
                md = smf.mixedlm(formula, d, groups=d['PATNO'], re_formula=re_formula)
                r = md.fit(method='lbfgs', reml=True)
                if not np.all(np.isfinite(r.bse.values)):
                    raise ValueError('non-finite standard errors')
                res = r
                note = '' if re_formula else 'random intercept only (slope model did not converge)'
                break
            except Exception as e:
                note = f'{type(e).__name__}'
        note = '; '.join(x for x in (note, drop_note) if x)
        if res is None:
            return lmm_blank(outcome, f'not estimable ({note})', n_neg, n_pos, len(d))

        b = res.fe_params
        V = res.cov_params().loc[b.index, b.index]
        names = list(b.index)
        it = 'time:saa' if 'time:saa' in names else 'saa:time'

        def contrast(weights):
            w = np.zeros(len(names))
            for k, v in weights.items():
                w[names.index(k)] = v
            est = float(w @ b.values)
            se = float(np.sqrt(w @ V.values @ w))
            return est, est - 1.96 * se, est + 1.96 * se, se

        s_neg = contrast({'time': 1.0})
        s_pos = contrast({'time': 1.0, it: 1.0})
        d_est, d_lo, d_hi, d_se = contrast({it: 1.0})
        pval = float(2 * stats.norm.sf(abs(d_est / d_se))) if d_se > 0 else np.nan

        return {'outcome': outcome, 'note': note,
                'n_participants': int(d.PATNO.nunique()), 'n_observations': int(len(d)),
                'n_neg': n_neg, 'n_pos': n_pos,
                'slope_neg': s_neg[0], 'slope_neg_lo': s_neg[1], 'slope_neg_hi': s_neg[2],
                'slope_pos': s_pos[0], 'slope_pos_lo': s_pos[1], 'slope_pos_hi': s_pos[2],
                'diff': d_est, 'diff_lo': d_lo, 'diff_hi': d_hi, 'p': pval, 'p_adj': np.nan,
                'covariates': ' + '.join(terms),
                '_fe': b, '_names': names, '_it': it, '_data': d, '_covm': V.values,
                '_xbar': pd.Series(np.asarray(res.model.exog).mean(axis=0),
                                   index=res.model.exog_names)}

    # one model per outcome, with `LMM_MT` applied across the outcomes
    lmm_res = {oc: lmm_fit_one(lmm_long, oc) for oc, _oname, _dir in LMM_OUTCOMES}
    ps = np.array([lmm_res[oc]['p'] for oc, _n, _d in LMM_OUTCOMES], dtype=float)
    ok = np.isfinite(ps)
    padj = np.full_like(ps, np.nan)
    if ok.sum() and LMM_MT != 'none':
        padj[ok] = multipletests(ps[ok], method=LMM_MT)[1]
    for i, (oc, _n, _d) in enumerate(LMM_OUTCOMES):
        lmm_res[oc]['p_adj'] = float(padj[i])

    lmm_tab = pd.DataFrame([{k: v for k, v in lmm_res[oc].items() if not k.startswith('_')}
                            for oc, _n, _d in LMM_OUTCOMES])
    lmm_tab.to_csv(LMM_OUT / 'lmm_saa_estimates.csv', index=False)

    # ---------------------------- markdown tables ----------------------------
    def lmm_row_md(first_col, r):
        padj = lmm_p(r.get('p_adj'))
        if np.isfinite(r.get('p_adj', np.nan)) and r['p_adj'] < LMM_ALPHA:
            padj = f'**{padj}**'
        return (f"| {first_col} | {r['n_participants']} ({r['n_neg']}/{r['n_pos']}) | "
                f"{lmm_fmt(r['slope_neg'], r['slope_neg_lo'], r['slope_neg_hi'])} | "
                f"{lmm_fmt(r['slope_pos'], r['slope_pos_lo'], r['slope_pos_hi'])} | "
                f"{lmm_fmt(r['diff'], r['diff_lo'], r['diff_hi'])} | "
                f"{lmm_p(r['p'])}{lmm_stars(r['p'])} | {padj} |")

    LMM_HDR = ('| n (neg/pos) | Slope, SAA-negative (95% CI) | Slope, SAA-positive (95% CI) | '
               'Difference positive − negative (95% CI) | P | P adj. |')

    lmm_md = ['## Table 1. Rate of clinical progression by baseline SAA status '
              '— all prodromal participants', '',
              '| Outcome ' + LMM_HDR, '|---|---|---|---|---|---|---|']
    for oc, oname, _dir in LMM_OUTCOMES:
        lmm_md.append(lmm_row_md(oname, lmm_res[oc]))

    lmm_ran = ('a random intercept and random slope for time per participant.'
               if LMM_RANDOM_SLOPE else 'a random intercept per participant.')
    lmm_codes = " or ".join(str(c) for c in LMM_CONV_CODES)
    if adjust:
        lmm_md += ['',
                   'Linear mixed models of each longitudinal outcome on time (years from baseline), '
                   'baseline SAA status (positive vs negative; SAA-negative is the reference), their '
                   'interaction, the outcome-specific covariates listed below, and phenoconversion during follow-up '
                   f'({LMM_CONV_COL} = {lmm_codes} at any visit'
                   + (', entered as a main effect and as an interaction with time)'
                      if LMM_CONV_TIME_INTERACTION else ', entered as a main effect)')
                   + ', with ' + lmm_ran]
    else:
        lmm_md += ['',
                   'Linear mixed models of each longitudinal outcome on time (years from baseline), '
                   'baseline SAA status (positive vs negative; SAA-negative is the reference), their '
                   'interaction, and the outcome-specific covariates listed below, with ' + lmm_ran,
                   '',
                   f'Participants who phenoconverted during follow-up ({LMM_CONV_COL} = '
                   f'{lmm_codes} at any visit within '
                   f'{LMM_TIME_MAX} years; n = {len(lmm_excl_ids)}) were excluded from all analyses.']
    lmm_md += ['',
               'Slopes are annual rates of change in the units of each outcome. The difference '
               'column is the interaction term: the extra annual change in SAA-positive '
               'participants relative to SAA-negative participants. Confidence intervals are '
               'Wald intervals from the fixed-effect covariance matrix.',
               '',
               f'P adj. = P value corrected across the {len(LMM_OUTCOMES)} outcomes '
               f'by the {LMM_MT.title()} procedure; values <{LMM_ALPHA} in bold.',
               '',
               f'Participants contributed to a model if they had at least {LMM_MIN_VISITS} '
               f'non-missing values of the outcome; a model was fitted only if at least '
               f'{LMM_MIN_PER_GROUP} participants were available in each SAA group.']
    lmm_bl_used = [on for oc, on, _d in LMM_OUTCOMES if lmm_bl_adj(oc)]
    if lmm_bl_used:
        lmm_md += ['', f'Models for {", ".join(lmm_bl_used)} are adjusted for the baseline value of '
                   f'the same outcome (value closest to baseline within {LMM_BL_WINDOW} years, '
                   'mean-centred; terms as listed below). In these models the visit supplying the '
                   'baseline value was not used as a response (post-baseline visits only), and '
                   f'participants needed a baseline value and at least {LMM_BL_MIN_FOLLOWUP} '
                   'post-baseline value of the outcome.']
    lmm_md += ['', 'Covariates by outcome:'] + [f'- {oname}: {lmm_terms_txt(oc, mode)}'
                                                for oc, oname, _d in LMM_OUTCOMES]
    (LMM_OUT / 'lmm_saa_estimates.md').write_text('\n'.join(lmm_md), encoding='utf-8')

    # ---------------------------- fitted trajectories ----------------------------
    def lmm_pred(r, saa, tgrid):
        """Marginal fitted mean at the covariate means, with a 95% Wald band."""
        b, names, it = r['_fe'], r['_names'], r['_it']
        # covariates at their mean in the data of THIS model; for a categorical
        # term C(x) this is the mean of each dummy
        xbar = r['_xbar']
        est, lo, hi = [], [], []
        for t in tgrid:
            w = np.array([float(xbar.get(n, 0.0)) for n in names])
            w[names.index('Intercept')] = 1.0
            w[names.index('time')] = t
            if 'saa' in names:
                w[names.index('saa')] = saa
            w[names.index(it)] = t * saa
            for n in names:                     # time:X terms (conv, bl_*) at the mean of X
                if n.startswith('time:') and n != it and n[5:] in names:
                    w[names.index(n)] = t * float(xbar[n[5:]])
            e = float(w @ b.values)
            se = float(np.sqrt(w @ r['_covm'] @ w))
            est.append(e); lo.append(e - 1.96 * se); hi.append(e + 1.96 * se)
        return np.array(est), np.array(lo), np.array(hi)

    lmm_tmax = float(LMM_TIME_MAX) if LMM_TIME_MAX is not None else float(
        np.nanmax(lmm_long.time.values)) if len(lmm_long) else 1.0
    lmm_tgrid = np.linspace(0, lmm_tmax, 40)
    LMM_LEG = [Line2D([0], [0], color=LMM_CNEG, lw=2, label='SAA-negative (reference)'),
               Line2D([0], [0], color=LMM_CPOS, lw=2, label='SAA-positive')]

    def lmm_panel(ax, r, title, show_n=True):
        if '_fe' not in r:
            ax.text(0.5, 0.5, r.get('note') or 'not estimable', ha='center', va='center',
                    transform=ax.transAxes, fontsize=12, color='#A32D2D', wrap=True)
            ax.set_xticks([]); ax.set_yticks([])
            ax.set_title(title, fontsize=12)
            return
        for saa, colr in ((0.0, LMM_CNEG), (1.0, LMM_CPOS)):
            e, lo, hi = lmm_pred(r, saa, lmm_tgrid)
            ax.plot(lmm_tgrid, e, color=colr, lw=1.6)
            ax.fill_between(lmm_tgrid, lo, hi, color=colr, alpha=0.13, linewidth=0)
        ax.set_xlim(0, lmm_tmax)
        sub = f'\nΔslope {r["diff"]:+.3f}/y, {lmm_ptxt(r["p"])}{lmm_stars(r.get("p_adj"))}'
        if show_n:
            sub += f'\nn = {r["n_neg"]}/{r["n_pos"]}'
        ax.set_title(title + sub, fontsize=12)

    # FIGURE 1 -- overall cohort, one panel per outcome
    fig, axes = plt.subplots(1, len(LMM_OUTCOMES),
                             figsize=(3.3 * len(LMM_OUTCOMES), 3.2), squeeze=False)
    for j, (oc, oname, _dir) in enumerate(LMM_OUTCOMES):
        ax = axes[0][j]
        lmm_panel(ax, lmm_res[oc], oname)
        ax.set_xlabel('Years from baseline')
        if j == 0:
            ax.set_ylabel('Model-estimated score')
    fig.legend(handles=LMM_LEG, loc='upper center', bbox_to_anchor=(0.5, 0.99),
               ncol=2, frameon=False, fontsize=12)
    fig.suptitle('Model-estimated trajectories by baseline SAA status — '
                 'all prodromal participants', fontsize=12, y=1.075)
    fig.tight_layout(rect=(0, 0, 1, 0.93))
    fig.savefig(LMM_OUT / 'fig_lmm_saa_trajectories.png', dpi=LMM_DPI)
    fig.savefig(LMM_OUT / 'fig_lmm_saa_trajectories.pdf')
    plt.close(fig)

    # ---------------------------- forest plots ----------------------------
    def lmm_forest(ax, rows, labels, xlabel, title):
        ys = np.arange(len(rows))[::-1]
        fin = [r for r in rows if np.isfinite(r['diff'])]
        span = max([abs(r['diff_lo']) for r in fin] + [abs(r['diff_hi']) for r in fin] + [0.01])
        for y, r, lab in zip(ys, rows, labels):
            if not np.isfinite(r['diff']):
                ax.text(0, y, r.get('note') or 'not estimable', ha='center', va='center',
                        fontsize=12, color='#A32D2D')
                continue
            pa = r.get('p_adj', np.nan)
            col = '#185FA5' if (np.isfinite(pa) and pa < LMM_ALPHA) else '#5F5E5A'
            ax.plot([r['diff_lo'], r['diff_hi']], [y, y], color=col, lw=1.5, solid_capstyle='butt')
            ax.plot(r['diff'], y, 'o', color=col, ms=5.5)
            ax.text(span * 1.28, y,
                    f"{r['diff']:+.3f} ({r['diff_lo']:+.3f} to {r['diff_hi']:+.3f})"
                    f"  {lmm_ptxt(r['p'])}{lmm_stars(pa)}",
                    va='center', fontsize=12, color=col)
        ax.axvline(0, color='#888780', ls='--', lw=0.8, zorder=0)
        ax.set_xlim(-span * 1.20, span * 3.4)
        ax.set_ylim(-0.7, len(rows) - 0.3)
        ax.set_yticks(ys)
        ax.set_yticklabels(labels, fontsize=8.5)
        ax.grid(axis='y', alpha=0)
        ax.set_xlabel(xlabel)
        ax.set_title(title, fontsize=12)

    # FIGURE 2 -- overall cohort, one row per outcome
    fig, ax = plt.subplots(figsize=(7.4, 0.62 * len(LMM_OUTCOMES) + 2.0))
    lmm_forest(ax, [lmm_res[oc] for oc, _n, _d in LMM_OUTCOMES],
               [oname for _oc, oname, _d in LMM_OUTCOMES],
               'Difference in annual slope, SAA-positive − SAA-negative (units/year)',
               'All prodromal participants   (reference = SAA-negative)')
    fig.suptitle('Difference in annual rate of change, SAA-positive vs SAA-negative',
                 fontsize=14, y=1.02)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    fig.savefig(LMM_OUT / 'fig_lmm_saa_slope_differences.png', dpi=LMM_DPI)
    fig.savefig(LMM_OUT / 'fig_lmm_saa_slope_differences.pdf')
    plt.close(fig)

    # ---------------------------- summary ----------------------------
    print('\n' + '\n'.join(lmm_md[:4 + len(LMM_OUTCOMES)]))
    lmm_notes = [f"{r['outcome']}: {r['note']}"
                 for r in lmm_tab.to_dict('records') if isinstance(r.get('note'), str) and r['note']]
    if lmm_notes:
        print('\nFit notes:')
        for c in lmm_notes:
            print('  ' + c)
    print('\nwritten to', LMM_OUT)

    return {'mode': mode, 'out': LMM_OUT, 'res': lmm_res, 'tab': lmm_tab,
            'n_all': lmm_d.PATNO.nunique(),
            'n_saa': n_saa, 'n_analysed': len(lmm_cov)}

## 1. Adjusted for phenoconversion

In [ ]:
run_adj = run_lmm_saa('adjust')

In [ ]:
lmm_n_flow(run_adj)

In [ ]:
lmm_n_per_year(run_adj)

## 2. Sensitivity analysis: phenoconverters excluded

In [ ]:
run_excl = run_lmm_saa('exclude')

In [ ]:
lmm_n_flow(run_excl)

In [ ]:
lmm_n_per_year(run_excl)